In [ ]:
import requests
import pyodbc
from datetime import datetime
import time

# ==============================
# 🔑 API CONFIG
# ==============================
API_KEY = "25a77923-5044-41ee-a0a0-1a1b654c7c52"
URL = f"https://api.cricapi.com/v1/currentMatches?apikey={API_KEY}"

# ==============================
# 🟡 SQL CONNECTION
# ==============================
conn = pyodbc.connect(
    "DRIVER={SQL Server};"
    "SERVER=FARAZ-PC\\SQLEXPRESS03;"
    "DATABASE=IPL_Data;"
    "Trusted_Connection=yes;"
)
cursor = conn.cursor()

# ==============================
# 🔴 MAIN LOOP
# ==============================
while True:
    try:
        response = requests.get(URL)
        data = response.json()

        matches = data.get("data", [])

        if not matches:
            print("❌ No matches from API")
            time.sleep(20)
            continue

        for match in matches:
            try:
                # ==============================
                # 🔥 IPL FILTER
                # ==============================
                series_name = str(match.get("series", "")).lower()
                match_name = str(match.get("name", "")).lower()

                if "ipl" not in series_name and "indian premier league" not in match_name:
                    continue

                # ==============================
                # 🔴 ONLY LIVE MATCH
                # ==============================
                if not match.get("matchStarted", False) or match.get("matchEnded", False):
                    continue

                print("🔥 LIVE IPL MATCH:", match.get("name"))

                match_id = match.get("id")

                teams = match.get("teams", [])
                team_a = teams[0] if len(teams) > 0 else "Team A"
                team_b = teams[1] if len(teams) > 1 else "Team B"

                score_list = match.get("score")

                # ==============================
                # 🟢 PROCESS SCORE
                # ==============================
                if score_list and isinstance(score_list, list):
                    for inning in score_list:

                        team_name = inning.get("inning", "Unknown")

                        if "Inning" in team_name:
                            team_name = team_name.split("Inning")[0].strip()

                        team_name = team_name.replace("Benguluru", "Bengaluru")

                        runs = int(inning.get("r", 0))
                        wickets = int(inning.get("w", 0))
                        overs = float(inning.get("o", 0))

                        # ==============================
                        # 🔥 UPSERT (UPDATE + INSERT)
                        # ==============================
                        cursor.execute("""
                        IF EXISTS (
                            SELECT 1 FROM IPL_Live_Data 
                            WHERE match_id = ? AND team = ?
                        )
                        BEGIN
                            UPDATE IPL_Live_Data
                            SET runs = ?, 
                                wickets = ?, 
                                overs = ?, 
                                timestamp = ?, 
                                match_status = ?
                            WHERE match_id = ? AND team = ?
                        END
                        ELSE
                        BEGIN
                            INSERT INTO IPL_Live_Data 
                            (match_id, team, runs, wickets, overs, batsman, non_striker, striker_flag, timestamp, match_time, match_status)
                            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                        END
                        """,
                        # EXISTS CHECK
                        match_id, team_name,

                        # UPDATE
                        runs, wickets, overs, datetime.now(), "Live",
                        match_id, team_name,

                        # INSERT
                        match_id, team_name, runs, wickets, overs,
                        "Unknown", "Unknown", 0,
                        datetime.now(), datetime.now(), "Live"
                        )

                else:
                    print("⚠️ Live but no score")

                    for team in [team_a, team_b]:
                        cursor.execute("""
                        IF EXISTS (
                            SELECT 1 FROM IPL_Live_Data 
                            WHERE match_id = ? AND team = ?
                        )
                        BEGIN
                            UPDATE IPL_Live_Data
                            SET timestamp = ?, match_status = ?
                            WHERE match_id = ? AND team = ?
                        END
                        ELSE
                        BEGIN
                            INSERT INTO IPL_Live_Data 
                            (match_id, team, runs, wickets, overs, batsman, non_striker, striker_flag, timestamp, match_time, match_status)
                            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                        END
                        """,
                        match_id, team,

                        datetime.now(), "Live",
                        match_id, team,

                        match_id, team, 0, 0, 0,
                        "Unknown", "Unknown", 0,
                        datetime.now(), datetime.now(), "Live"
                        )

                conn.commit()
                print("✅ Data Updated (No duplicates)")

            except Exception as e:
                print("⚠️ Match error:", e)

    except Exception as e:
        print("❌ API Error:", e)

    time.sleep(20)